# Titanic Tutorial (Colab — Kaggle 다운로드 버전)

Kaggle의 [Titanic Tutorial](https://www.kaggle.com/code/alexisbcook/titanic-tutorial) (by Alexis Cook) 을 **Google Colab에서 실행**하는 노트북입니다.

- 실행 시 **Kaggle API로 대회 데이터를 직접 다운로드**합니다 (`train.csv`, `test.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` 가 `/content` 에 생성됩니다.

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Titanic 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

# 대회 데이터 다운로드 + 압축 해제
api.competition_download_files("titanic", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 불러오기

In [ ]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

train_data = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
train_data.head()

In [ ]:
test_data = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
test_data.head()

## 3. 패턴 살펴보기 (성별에 따른 생존율)

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women) / len(women)

print("% of women who survived:", rate_women)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men) / len(men)

print("% of men who survived:", rate_men)

## 4. 랜덤 포레스트 모델 학습 & 제출 파일 생성

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

submission_path = os.path.join(WORK_DIR, "submission.csv")
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv(submission_path, index=False)
print("Your submission was successfully saved to:", submission_path)

## 5. 제출 파일 내려받기

`submission.csv` 가 `/content` 에 생성되었습니다. 좌측 파일창에서 받거나 아래 셀로 바로 내려받으세요.

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Titanic 제출 페이지](https://www.kaggle.com/c/titanic/submit)**

- 대회 홈: https://www.kaggle.com/c/titanic
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.